# 個人情報をマスクする (仮名化)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/15_ner/pii_mask.ipynb)

NLP クックブック 15-07 ／ 固有表現抽出

テキスト中の人名・地名・電話番号・メールアドレスなどを伏せ字にして、データを共有できる形にしたい。

In [1]:
!pip install -q ginza ja_ginza

In [2]:
import re
import spacy

nlp = spacy.load("ja_ginza")

REGEX = {
    "EMAIL": r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+",
    "PHONE": r"0\d{1,4}-\d{1,4}-\d{4}",
    "POSTAL": r"〒?\d{3}-\d{4}",
}
NER_LABELS = {"Person": "PERSON", "City": "LOCATION", "Province": "LOCATION", "Company": "ORG", "School": "ORG"}

def mask(text):
    spans = []
    for label, pat in REGEX.items():
        spans += [(m.start(), m.end(), label) for m in re.finditer(pat, text)]
    for ent in nlp(text).ents:
        if ent.label_ in NER_LABELS:
            spans.append((ent.start_char, ent.end_char, NER_LABELS[ent.label_]))
    spans.sort(key=lambda s: (s[0], -(s[1] - s[0])))
    out, pos = [], 0
    for start, end, label in spans:
        if start < pos:      # 重なりは先に始まる (長い) ほうを優先
            continue
        out += [text[pos:start], f"<{label}>"]
        pos = end
    return "".join(out) + text[pos:]

text = "山田太郎(yamada@example.com、090-1234-5678)は横浜市在住で、ソニーに勤務している。佐藤花子は東京大学の学生です。"
print(mask(text))

<PERSON>(<EMAIL>、<PHONE>)は<LOCATION>在住で、<ORG>に勤務している。<PERSON>は<ORG>の学生です。


- NER の再現率は 100% ではない。公開前には人手の確認が必要。再現率を優先するなら複数のモデルの和集合をとる。
- 単にマスクすると文脈が失われる。「山田太郎 → 人物A」のように同じ実体に同じ仮名を割り当てると、後段の分析に使いやすい。
- 英語を含む多言語なら Microsoft Presidio (`presidio-analyzer`) が定番。